In [ ]:
# pipeline for this simsearch model
# what particularly simsearch does is that, 
# it's a self supervised learning model that learns about images from
# unlabelled dataset using embedding vector space and 
# when it came up with new images, it tries to match similarity
# of the new images with old images
# search similarity in embedding space

# project pipeline
# img -> self-supervised training -> encoder network -> img embeddings -> vector db -> similarity search

# SimSearch model training pipeline
# img -> augmentation -> encoder -> embeddings -> contrastive loss


In [9]:
# device configuration for model training
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)


cpu


In [ ]:
# importing all necessary items for model training
from torch.utils.data import DataLoader

from models.model import SimSearchModel
from data.dataset import SimSearchDataset
from utils.augmentations import SimSearch_transform
from losses.nt_xnt import nt_xnt_loss


# creating dataset for model training
dataset = SimSearchDataset(
    root_dir="mammals",
    transform=SimSearch_transform
)

# dataloader for loading the data in model training
loader = DataLoader(
    dataset=dataset,
    batch_size=32,
    shuffle=True
)

# SimSearchDataset returns view1, view2 (2 augmentation of same image)
# view1, view2 goes into DataLoader for loading into model


In [15]:
simsearchmodel = SimSearchModel().to(device=device)  # loading model into device

# optimizer for model training
optimizer=torch.optim.AdamW(
    simsearchmodel.parameters(),
    lr=3e-4,
    weight_decay=1e-4
)

In [ ]:
# SimSearch model training
def SimSearch_training(model, loader, loss_fn, optimizer_fn, device, epochs):
    
    for epoch in range(epochs):
        for view1, view2 in loader: # iterating through loader
            view1=view1.to(device) # loading view1 into device, similarly for view2
            view2=view2.to(device)

            z1=model(view1) # embedding for view1
            z2=model(view2) # embedding for view2

            loss=loss_fn(z1,z2) # calculating loss
            optimizer_fn.zero_grad() # zero gradient of optimizer
            loss.backward() # backpropagation
            optimizer_fn.step() # usign optimizer function
        print(f"Epoch: {epoch}| Loss: {loss.item()}")


SimSearch_training(model=simsearchmodel,loader=loader,loss_fn=nt_xnt_loss,optimizer_fn=optimizer,device=device,epochs=20)

# training needs a good compute power
# need something for it


KeyboardInterrupt: 